# Classification: Bagging Classifier

## Justification of Preprocessing Strategy

### Scale Invariance of Tree-Based Ensembles

The Bagging Classifier in this notebook uses our optimized Decision Tree as its base estimator. Since Decision Trees partition data using feature thresholds rather than distance metrics, they are **scale-invariant**. Consequently, the model's predictive power remains identical regardless of whether the data is standardized, normalized, or in its original form. To maintain computational efficiency and clinical interpretability, we will use the **Original Data**.

## Ensemble of Optimized Base Estimators

Instead of using default settings, we are initializing the Bagging process with the "Champion" parameters found in your previous Decision Tree experiments:
- **Criterion**: entropy
- **Max Depth**: 29
- **Min Samples Leaf**: 3
- **Min Samples Split**: 7

This ensures that each individual model in the ensemble is already highly tuned to the diabetes dataset, while the Bagging process adds robustness by reducing variance through random sampling.

## Experiment Design

We will run 3 optimization stages to evaluate the impact of ensemble size and sampling density on our optimized trees:

1. **Baseline**: Bagging with 50 estimators and 80% sample size, using the Champion Decision Tree as the base.
2. **GridSearchCV**: Exhaustive search over n_estimators and max_samples to find the best ensemble configuration for Recall.
3. **Optuna**: Bayesian optimization to further refine the number of estimators and bootstrap strategy.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

#MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_Bagging")

2026/05/04 13:01:35 INFO mlflow.tracking.fluent: Experiment with name 'Classification_Bagging' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/EnsembleMethods/Bagging/BaggingClassifier/mlruns/7'), creation_time=1777896095819, experiment_id='7', last_update_time=1777896095819, lifecycle_stage='active', name='Classification_Bagging', tags={}, workspace='default'>

In [ ]:
df = pd.read_csv("C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/data/diabetes_dataset_new_variables.csv")

# Categorical columns for One-Hot Encoding
categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# DEFINING THE CHAMPION ESTIMATOR (From our previous DT results)
champion_dt = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=29,
    min_samples_leaf=3,
    min_samples_split=7,
    random_state=42
)

def log_bagging_metrics(y_true, y_pred, duration):
    """Utility function to log metrics to MLflow"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="Bagging_Baseline"):
    # Using optimized base estimator with worksheet configurations
    bag_base = BaggingClassifier(
        estimator=champion_dt,
        n_estimators=50,
        max_samples=0.8,
        bootstrap=True,
        random_state=42,
        n_jobs=-1 
    )
    
    start_time = time.time()
    bag_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred = bag_base.predict(X_test)
    
    mlflow.log_params(bag_base.get_params())
    mlflow.log_param("base_estimator_type", "Optimized_DT")
    mlflow.log_param("optimization", "none")
    log_bagging_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="Bagging_GridSearch"):
    # Grid search focusing on ensemble size and sample ratio
    param_grid = {
        'n_estimators': [10, 50, 100],
        'max_samples': [0.6, 0.8, 1.0]
    }
    
    grid = GridSearchCV(
        BaggingClassifier(estimator=champion_dt, random_state=42),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred_grid = grid.best_estimator_.predict(X_test)
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("base_estimator_type", "Optimized_DT")
    mlflow.log_param("optimization", "GridSearchCV")
    log_bagging_metrics(y_test, y_pred_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 150),
        "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42
    }
    
    model = BaggingClassifier(
        estimator=champion_dt,
        **params,
        n_jobs=-1
    )
    
    # Using 3-fold CV for search efficiency
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="Bagging_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=10) 
    duration = time.time() - start_time
    
    # Training the best ensemble found by Optuna
    best_bag = BaggingClassifier(
        estimator=champion_dt,
        **study.best_params,
        n_jobs=-1
    )
    best_bag.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("base_estimator_type", "Optimized_DT")
    mlflow.log_param("optimization", "optuna")
    log_bagging_metrics(y_test, best_bag.predict(X_test), duration)

[I 2026-05-04 13:04:01,823] A new study created in memory with name: no-name-ff03d444-9431-4bae-bb1b-5cb39e3779d7
[I 2026-05-04 13:04:22,960] Trial 0 finished with value: 0.8693486923661896 and parameters: {'n_estimators': 66, 'max_samples': 0.8690197523751262, 'bootstrap': False}. Best is trial 0 with value: 0.8693486923661896.
[I 2026-05-04 13:04:44,297] Trial 1 finished with value: 0.8689945126778756 and parameters: {'n_estimators': 93, 'max_samples': 0.999150115975147, 'bootstrap': True}. Best is trial 0 with value: 0.8693486923661896.
[I 2026-05-04 13:04:58,013] Trial 2 finished with value: 0.8690361806467072 and parameters: {'n_estimators': 65, 'max_samples': 0.8277730643094032, 'bootstrap': True}. Best is trial 0 with value: 0.8693486923661896.
[I 2026-05-04 13:05:13,267] Trial 3 finished with value: 0.8690570152822051 and parameters: {'n_estimators': 76, 'max_samples': 0.7887009511245995, 'bootstrap': True}. Best is trial 0 with value: 0.8693486923661896.
[I 2026-05-04 13:05:35

## Runs Summary

| Run | Optimization | Bootstrap | n_estimators | max_samples | Accuracy | F1 | Recall | Fit Time |
|---|---|---|---:|---:|---|---:|---:|---:|
| Bagging_Baseline | none | True | 50 | 0.8 | 0.91995 | 0.92854 | 0.86675 | 10.62s |
| Bagging_GridSearch | GridSearchCV | True | 10 | 0.8 | 0.91695 | 0.92617 | 0.86825 | 134.19s |
| Bagging_Optuna | optuna | False | 66 | 0.8690197523751262 | 0.91935 | 0.92806 | 0.86708 | 194.73s |

### Additional logged parameters
- `random_state = 42` in all runs
- `n_jobs = -1` in all runs
- `base_estimator_type = Optimized_DT` in all runs
- `bootstrap_features = False` in the baseline run
- `oob_score = False` in the baseline run
- Other Bagging parameters such as `max_features`, `verbose`, and `warm_start` remained at their default values

## Best Run Justification for Streamlit

A melhor run para usar no Streamlit é a **Bagging_Baseline**. Embora o **Bagging_GridSearch** tenha o maior recall, a diferença face ao baseline é muito pequena ($0.86825$ vs. $0.86675$), enquanto o custo em tempo de treino cresce de forma muito significativa. O **Bagging_Optuna** também não supera o baseline de forma convincente: melhora ligeiramente o recall, mas perde em accuracy, F1 e tempo de execução.

Para um caso clínico como este, o objetivo não é apenas maximizar uma métrica isolada, mas manter um bom equilíbrio entre identificar corretamente pacientes com diabetes e evitar falsos alarmes. Nesse equilíbrio, o baseline é o melhor compromisso porque apresenta:
- a melhor **accuracy**;
- o melhor **F1-score**;
- um **recall** muito próximo dos restantes;
- o menor **tempo de treino**, o que é mais prático para integração e manutenção no Streamlit.

